<h1><center>Recommender Systems YSDA Course!</center></h1>
<h1><center>Семинар №2</center></h1>

<center><img src="https://github.com/yandexdataschool/recsys_course/blob/2026_spring/week02_candgen/logo.jpg?raw=1" width="500" /></center>

**В этом семинаре мы:**
- Познакомимся с датасетом YAMBDA
- Ссылка на оригинальный датасет: https://huggingface.co/datasets/yandex/yambda
- Посмотрим на контест курса: https://www.kaggle.com/competitions/ysda-rec-sys-2026
- Напишем бейзлайн
- Обучим более сложные модели (CatBoost)
- Напишем несколько новых метрик оценки качества ранжирования

**Баллы за пороги:**
- 5 баллов за пробитие 0.06
- 10 баллов за ...
- 15 баллов за ...
- Топ 3 - дополнительные 10 баллов
- Топ 10 - дополнительные 5 баллов

In [1]:
import numpy as np
import polars as pl
import seaborn as sns
import matplotlib.pyplot as plt

from catboost import CatBoostClassifier, Pool

import gc
from pathlib import Path
import os
import logging
from copy import deepcopy

import numpy as np
import polars as pl
import seaborn as sns
import matplotlib.pyplot as plt

from catboost import CatBoostRanker, Pool, CatBoostClassifier
import torch
from utils.train_utils import (
    train_and_test_catboost,
    check_on_dataset,
    check_on_sampled,
    generate_features_and_negatives_local,
    generate_test_features_local,
    METRICS,
    TO_DROP,
)
from utils.utils import (
    get_dataset,
    get_test_users,
    PREPROCESSED_DIR,
    MODEL_PATH,
    FINAL_MODEL_PATH,
    recall_at_k,
)


import kagglehub
from kagglehub import KaggleDatasetAdapter


logging.basicConfig(level=logging.WARN)

# 🗄 Датасет:

In [26]:
import duckdb
from utils.yambda_dataset import YambdaDataset
from utils.utils import setup_duckdb_connection

APPROX_TEST_USERS_ONLY = True
if APPROX_TEST_USERS_ONLY:
    FULL_DATA_PATH = f"{PREPROCESSED_DIR}/initial_data_test_users_10.parquet"
    SAMPLED_DATA_PATH = f"{PREPROCESSED_DIR}/initial_data_test_users_sample_10.parquet"
else:
    FULL_DATA_PATH = f"{PREPROCESSED_DIR}/initial_data_10.parquet"
    SAMPLED_DATA_PATH = f"{PREPROCESSED_DIR}/initial_data_sample_10.parquet"

# Fraction of "listen" events to keep (to avoid memory explosion)
LISTEN_SAMPLE_FRACTION = 0.03
SAMPLE_PERCENT = 5


In [27]:
data_sampled = get_dataset(
    full_data_path=FULL_DATA_PATH,
    sampled_data_path=SAMPLED_DATA_PATH,
    approx_test_users_only=APPROX_TEST_USERS_ONLY,
    listen_sample_fraction=LISTEN_SAMPLE_FRACTION,
    deduplicate=True,
    invalidate_cache=False,
    sample_percent=SAMPLE_PERCENT,
)

data = (
    data_sampled
    # .filter(pl.col("event_type") == "like")
    # .sample(500_000, seed=42)
    .with_columns(
        pl.when(pl.col("event_type").eq("like"))
        .then(pl.lit(1))
        .otherwise(pl.lit(0))
        .alias("target")
    ).select(
        [
            "uid",
            "item_id",
            "timestamp",
            "is_organic",
            "event_type",
            "target",
        ]
    )
)

read parquet


In [28]:
# import kagglehub
# from kagglehub import KaggleDatasetAdapter

# file_path = "likes.parquet"

# data = (
#     kagglehub.dataset_load(
#         KaggleDatasetAdapter.POLARS,
#         "thekabeton/ysda-recsys-2026-yambda-dataset/versions/3",
#         file_path,
#     )
#     .collect()
#     # .sample(5_000_000, seed=42)
#     .sample(500_000, seed=42)
# )  # Ограничение для семинара, лучше использовать все данные


In [29]:
file_path = "test_users.csv"

test_users = kagglehub.dataset_load(
    KaggleDatasetAdapter.POLARS,
    "thekabeton/ysda-recsys-2026-yambda-dataset/versions/3",
    file_path,
).collect()


In [30]:
file_path = "artist_item_mapping_small.parquet"

artists = kagglehub.dataset_load(
    KaggleDatasetAdapter.POLARS,
    "thekabeton/ysda-recsys-2026-yambda-dataset/versions/3",
    file_path,
).collect()


In [31]:
SPLIT_1 = 100
SPLIT_2 = 200
SPLIT_3 = 210

SECONDS_IN_DAY = 60 * 60 * 24


def make_parts(
    data, day_split_1: int = 100, day_split_2: int = 200, day_split_3: int = 300
):
    data_part1 = data.filter(pl.col("timestamp") < SECONDS_IN_DAY * day_split_1)
    data_part2 = data.filter(
        (pl.col("timestamp") >= SECONDS_IN_DAY * day_split_1)
        & (pl.col("timestamp") < SECONDS_IN_DAY * day_split_2)
    )
    data_part3 = data.filter(
        (pl.col("timestamp") >= SECONDS_IN_DAY * day_split_2)
        & (pl.col("timestamp") < SECONDS_IN_DAY * day_split_3)
    )
    return data_part1, data_part2, data_part3

# 🦾 CatBoost

<center><img src="https://github.com/yandexdataschool/recsys_course/blob/2026_spring/week02_candgen/Timesplit1.svg?raw=1" width="1100" /></center>


Давайте соберём какие-то фичи из данных и обучим на них градиентный бустинг. Нужно не забывать про временные лики. Нельзя давать модели видеть данные из будущего, поэтому фичи для каждого семпла должны быть посчитаны на данных из прошлого. В простейшей схеме предлагается разделить размеченые данные на 3 части:
- Вторая часть - train
- Третья часть - validation
- Первую часть используем для расчёта статистик для трейна
- Для валидации считаем статистики используя первую и вторую части вместе

#### Делим data на 3 части:

In [32]:
data_part1, data_part2, data_part3 = make_parts(
    data, day_split_1=SPLIT_1, day_split_2=SPLIT_2, day_split_3=SPLIT_3
)

#### Научимся определять decay для ивента

In [33]:
def get_event_oldness(df: pl.DataFrame):
    # decay 2 times each week
    decay_factor = 1.0 / 2.0
    decay_interval = 60 * 60 * 24 * 7

    m = df["timestamp"].max()
    return df.with_columns(
        pl.when((m - pl.col("timestamp")) / decay_interval > 1)
        .then(decay_factor ** ((m - pl.col("timestamp")) / decay_interval))
        .otherwise(pl.lit(1.0))
        .alias("decay_factor")
    )

#### Набираем негативы:

In [ ]:
def add_popular_random_negatives(
    pool_df: pl.DataFrame,
    df: pl.DataFrame,
    k: int,
    top_n: int = 100_000,
    seed: int = 42,
) -> pl.DataFrame:
    n = df.height * k

    top_items = (
        get_event_oldness(pool_df)
        .unique(["item_id", "uid"])
        .group_by("item_id")
        .agg(pl.col("decay_factor").sum().alias("popularity_decayed"))
        .sort("popularity_decayed", descending=True)
        .head(top_n)
        .select(["item_id"])
    )

    # top_items = (
    #     pool_df.unique(['item_id', 'uid']).group_by("item_id").len()
    #     .sort("len", descending=True)
    #     .head(top_n)
    #     .select("item_id")
    # )

    neg = pl.DataFrame(
        {
            "uid": pl.concat([df.get_column("uid")] * k, rechunk=True),
            "item_id": top_items.get_column("item_id").sample(
                n=n, with_replacement=True, seed=seed
            ),
            # "event_type": pl.repeat("random_negative", n, eager=True),
            "target": pl.repeat(0, n, eager=True),
        }
    )

    pos = df.select(
        [
            "uid",
            "item_id",
            #  "even_type",
            "target",
        ]
    ).with_columns(
        pl.lit(k).alias("weight"),
    )
    neg_filtered = neg.join(pos, on=("uid", "item_id"), how="anti").with_columns(
        pl.lit(1).alias("weight")
    )

    return pl.concat([pos, neg_filtered], how="vertical")


In [39]:
def add_item_popularity(train: pl.DataFrame, df: pl.DataFrame) -> pl.DataFrame:
    pop = train.group_by("item_id").len().rename({"len": "item_popularity"})
    return df.join(pop, on="item_id", how="left").with_columns(
        pl.col("item_popularity").fill_null(0)
    )


def add_item_popularity_with_decay(
    train: pl.DataFrame, df: pl.DataFrame
) -> pl.DataFrame:
    train_ = get_event_oldness(train)
    pop = train_.group_by("item_id").agg(
        pl.col("decay_factor").sum().alias("item_popularity_decayed")
    )
    return df.join(pop, on="item_id", how="left").with_columns(
        pl.col("item_popularity_decayed").fill_null(0)
    )


def add_user_count_likes(train: pl.DataFrame, df: pl.DataFrame) -> pl.DataFrame:
    pop = train.group_by("uid").len().rename({"len": "user_count_likes"})
    return df.join(pop, on="uid", how="left").with_columns(
        pl.col("user_count_likes").fill_null(0)
    )


def add_user_count_likes_with_decay(
    train: pl.DataFrame, df: pl.DataFrame
) -> pl.DataFrame:
    train_ = get_event_oldness(train)
    pop = train_.group_by("uid").agg(
        pl.col("decay_factor").sum().alias("user_count_likes_decayed")
    )
    return df.join(pop, on="uid", how="left").with_columns(
        pl.col("user_count_likes_decayed").fill_null(0)
    )


def add_artist_count_likes(
    train: pl.DataFrame, df: pl.DataFrame, item2artist: pl.DataFrame
) -> pl.DataFrame:
    stats = (
        train.select(["item_id"])
        .join(item2artist, on="item_id", how="left")
        .group_by(["artist_id"])
        .len()
        .rename({"len": "artist_like_cnt"})
    )

    return (
        df.join(item2artist, on="item_id", how="left")
        .join(stats, on="artist_id", how="left")
        .with_columns(pl.col("artist_like_cnt").fill_null(0))
        .drop("artist_id")
    )


def add_artist_count_likes_with_decay(
    train: pl.DataFrame, df: pl.DataFrame, item2artist: pl.DataFrame
) -> pl.DataFrame:
    train_ = get_event_oldness(train)
    stats = (
        train_.select(["item_id", "decay_factor"])
        .join(item2artist, on="item_id", how="left")
        .group_by(["artist_id"])
        .agg(pl.col("decay_factor").sum().alias("artist_like_cnt_decayed"))
    )

    return (
        df.join(item2artist, on="item_id", how="left")
        .join(stats, on="artist_id", how="left")
        .with_columns(pl.col("artist_like_cnt_decayed").fill_null(0))
        .drop("artist_id")
    )


def add_item_organic_share(train: pl.DataFrame, df: pl.DataFrame) -> pl.DataFrame:
    share = train.group_by("item_id").agg(
        pl.col("is_organic").mean().alias("item_organic_share")
    )
    return df.join(share, on="item_id", how="left").with_columns(
        pl.col("item_organic_share").fill_null(0)
    )


def add_item_organic_share_with_decay(
    train: pl.DataFrame, df: pl.DataFrame
) -> pl.DataFrame:
    train_ = get_event_oldness(train)
    share = train_.group_by("item_id").agg(
        (pl.col("is_organic") * pl.col("decay_factor"))
        .mean()
        .alias("item_organic_share_decayed")
    )
    return df.join(share, on="item_id", how="left").with_columns(
        pl.col("item_organic_share_decayed").fill_null(0)
    )


def add_user_organic_share(train: pl.DataFrame, df: pl.DataFrame) -> pl.DataFrame:
    share = train.group_by("uid").agg(
        pl.col("is_organic").mean().alias("user_organic_share")
    )
    return df.join(share, on="uid", how="left").with_columns(
        pl.col("user_organic_share").fill_null(0)
    )


def add_user_organic_share_with_decay(
    train: pl.DataFrame, df: pl.DataFrame
) -> pl.DataFrame:
    train_ = get_event_oldness(train)
    share = train_.group_by("uid").agg(
        (pl.col("is_organic") * pl.col("decay_factor"))
        .mean()
        .alias("user_organic_share_decayed")
    )
    return df.join(share, on="uid", how="left").with_columns(
        pl.col("user_organic_share_decayed").fill_null(0)
    )


def add_artist_organic_share(
    train: pl.DataFrame, df: pl.DataFrame, item2artist: pl.DataFrame
) -> pl.DataFrame:
    stats = (
        train.join(item2artist, on="item_id", how="left")
        .group_by(["artist_id"])
        .agg(pl.col("is_organic").mean().alias("artist_organic_share"))
    )

    return (
        df.join(item2artist, on="item_id", how="left")
        .join(stats, on="artist_id", how="left")
        .with_columns(pl.col("artist_organic_share").fill_null(0))
        .drop("artist_id")
    )


def add_artist_organic_share_with_decay(
    train: pl.DataFrame, df: pl.DataFrame, item2artist: pl.DataFrame
) -> pl.DataFrame:
    train_ = get_event_oldness(train)
    stats = (
        train_.join(item2artist, on="item_id", how="left")
        .group_by(["artist_id"])
        .agg(
            (pl.col("is_organic") * pl.col("decay_factor"))
            .mean()
            .alias("artist_organic_share_decayed")
        )
    )

    return (
        df.join(item2artist, on="item_id", how="left")
        .join(stats, on="artist_id", how="left")
        .with_columns(pl.col("artist_organic_share_decayed").fill_null(0))
        .drop("artist_id")
    )


def add_user_artist_like_cnt(
    events: pl.DataFrame, df: pl.DataFrame, item2artist: pl.DataFrame
) -> pl.DataFrame:
    stats = (
        events.select(["uid", "item_id"])
        .join(item2artist, on="item_id", how="left")
        .group_by(["uid", "artist_id"])
        .len()
        .rename({"len": "user_artist_like_cnt"})
    )

    return (
        df.join(item2artist, on="item_id", how="left")
        .join(stats, on=["uid", "artist_id"], how="left")
        .with_columns(pl.col("user_artist_like_cnt").fill_null(0))
        .drop("artist_id")
    )


def add_user_artist_like_cnt_with_decay(
    events: pl.DataFrame, df: pl.DataFrame, item2artist: pl.DataFrame
) -> pl.DataFrame:
    events_ = get_event_oldness(events)
    stats = (
        events_.select(["uid", "item_id", "decay_factor"])
        .join(item2artist, on="item_id", how="left")
        .group_by(["uid", "artist_id"])
        .agg(pl.col("decay_factor").sum().alias("user_artist_like_cnt_decayed"))
    )

    return (
        df.join(item2artist, on="item_id", how="left")
        .join(stats, on=["uid", "artist_id"], how="left")
        .with_columns(pl.col("user_artist_like_cnt_decayed").fill_null(0))
        .drop("artist_id")
    )


In [40]:
def add_features(
    history: pl.DataFrame,
    to_enrich: pl.DataFrame,
    artists: pl.DataFrame,
    add_negatives: bool = True,
    filter_likes: bool = False,
):
    if filter_likes:
        to_enrich = to_enrich.filter(pl.col("target").eq(pl.lit(1)))
        history = history.filter(pl.col("target").eq(pl.lit(1)))
    if add_negatives:
        to_enrich = add_popular_random_negatives(history, to_enrich, 10)

    to_enrich = add_item_popularity(history, to_enrich)
    # to_enrich = add_item_popularity_with_decay(history, to_enrich)
    to_enrich = add_artist_count_likes(history, to_enrich, artists)
    # to_enrich = add_artist_count_likes_with_decay(history, to_enrich, artists)
    # to_enrich = add_item_organic_share(history, to_enrich)
    # to_enrich = add_item_organic_share_with_decay(history, to_enrich)
    # to_enrich = add_user_organic_share(history, to_enrich)
    # to_enrich = add_user_organic_share_with_decay(history, to_enrich)
    # to_enrich = add_artist_organic_share(history, to_enrich, artists)
    # to_enrich = add_artist_organic_share_with_decay(history, to_enrich, artists)
    to_enrich = add_user_artist_like_cnt(history, to_enrich, artists)
    # to_enrich = add_user_artist_like_cnt_with_decay(history, to_enrich, artists)
    return to_enrich

#### Проделываем то-же самое для валидации. Фичи считаем по событиям из 2 части датасета. Затем клеим их к 3 части:

In [41]:
ADD_NEGATIVES = True
FILTER_LIKES = True
train = add_features(
    data_part1,
    data_part2,
    artists,
    add_negatives=ADD_NEGATIVES,
    filter_likes=FILTER_LIKES,
)
val = add_features(
    data_part2,
    data_part3,
    artists,
    add_negatives=ADD_NEGATIVES,
    filter_likes=FILTER_LIKES,
)

In [ ]:
# from utils.data_preprocessing_pipeline import DataPreprocessingPipeline

# user_organic_share: 62.84630738306897
# user_artist_like_cnt: 10.44752639806736
# item_popularity: 9.210531430652539
# artist_organic_share: 7.138556854849871
# artist_like_cnt: 6.82155358516451
# item_organic_share: 3.5355243481967467

# fstr_dict = {
#     "item_all_life": 14.999871445868624,
#     "item_all_30d": 8.318400888547968,
#     "item_likes_life": 3.203577265056065,
#     "uid_like_all_ratio_90d": 0.7277936238767376,
#     "item_cnt_30d": 0.11285190197427988,
#     "item_org_likes_7d": 0.10117153065597632,
#     "uid_artist_like_ratio_30d": 0.05095618463919989,
#     "artist_id": 0.0,
#     "album_id": 0.0,
#     "item_listens_life": 0.0,
#     "item_org_all_life": 1.0,
# }
# feature_columns = list(fstr_dict.keys())
# # fstr_dict = None

# pipeline = DataPreprocessingPipeline(
#     random_negatives_configs=[],
#     feature_windows=["7d", "30d", "90d"],
#     feature_importances=fstr_dict,
#     feature_importance_threshold=0.01,
#     target_for_fake=0.00,
#     preprocess_dir=".tmp",
#     preprocess_prefix="test_pipeline_consistency",
#     use_cache=False,
#     invalidate_cache=True,
#     seed=42,
# )

# data = data.filter(pl.col("target").eq(1))
# data_part1, data_part2, data_part3 = make_parts(
#     data, day_split_1=SPLIT_1, day_split_2=SPLIT_2, day_split_3=SPLIT_3
# )
# with_negatives_2 = add_popular_random_negatives(data_part1, data_part2, 10)
# with_negatives_3 = add_popular_random_negatives(data_part2, data_part3, 10)

# all_with_negatives = pl.concat(
#     [data_part1, with_negatives_2, with_negatives_3], how="vertical"
# )



#### Обучаем катбуст:

In [44]:
MAX_GROUP_SIZE = 1023
train = train.group_by("uid").head(MAX_GROUP_SIZE)

to_drop = ["target", "timestamp", "item_id", "uid", "is_organic"] + (
    ["weight"] if ADD_NEGATIVES else []
)
to_drop = [column for column in to_drop if column in train.columns]
train_pool = Pool(
    data=train.drop(to_drop),
    label=train["target"],
    group_id=train["uid"],
)

val = val.group_by("uid").head(MAX_GROUP_SIZE)
val_pool = Pool(
    data=val.drop(to_drop),
    label=val["target"],
    group_id=val["uid"],
)

In [45]:
model = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.1,
    depth=6,
    l2_leaf_reg=10,
    loss_function="CrossEntropy",
    eval_metric="AUC",
    custom_metric=[
        "AUC",
        "CrossEntropy",
        "Precision",
        "Recall",
        "Accuracy",
        "QueryAUC",
        "RecallAt",
        "LogLikelihoodOfPrediction",
    ],
    early_stopping_rounds=100,
    verbose=10,
    use_best_model=True,
    task_type="GPU",
)

model.fit(train_pool, eval_set=val_pool, plot=True)

MetricVisualizer(layout=Layout(align_self='stretch', height='500px'))

Default metric period is 5 because AUC, LogLikelihoodOfPrediction, QueryAUC, RecallAt is/are not implemented for GPU
Metric QueryAUC is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Metric RecallAt is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Metric LogLikelihoodOfPrediction is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


0:	test: 0.8791092	best: 0.8791092 (0)	total: 75.4ms	remaining: 1m 15s
10:	test: 0.8887853	best: 0.8890716 (7)	total: 213ms	remaining: 19.1s
20:	test: 0.8892254	best: 0.8892254 (20)	total: 341ms	remaining: 15.9s
30:	test: 0.8903089	best: 0.8903089 (30)	total: 473ms	remaining: 14.8s
40:	test: 0.8903351	best: 0.8903759 (35)	total: 597ms	remaining: 14s
50:	test: 0.8903804	best: 0.8903854 (49)	total: 725ms	remaining: 13.5s
60:	test: 0.8903188	best: 0.8903982 (52)	total: 853ms	remaining: 13.1s
70:	test: 0.8903120	best: 0.8903982 (52)	total: 983ms	remaining: 12.9s
80:	test: 0.8903029	best: 0.8903982 (52)	total: 1.11s	remaining: 12.6s
90:	test: 0.8902918	best: 0.8903982 (52)	total: 1.23s	remaining: 12.3s
100:	test: 0.8902932	best: 0.8903982 (52)	total: 1.36s	remaining: 12.1s
110:	test: 0.8902857	best: 0.8903982 (52)	total: 1.48s	remaining: 11.9s
120:	test: 0.8902842	best: 0.8903982 (52)	total: 1.61s	remaining: 11.7s
130:	test: 0.8902795	best: 0.8903982 (52)	total: 1.73s	remaining: 11.5s
140:	

CatBoostClassifier(custom_metric=['AUC', 'CrossEntropy', 'Precision', 'Recall', 'Accuracy', 'QueryAUC', 'RecallAt', 'LogLikelihoodOfPrediction'], depth=6, early_stopping_rounds=100, eval_metric='AUC', iterations=1000, l2_leaf_reg=10, learning_rate=0.1, loss_function='CrossEntropy', task_type='GPU', use_best_model=True, verbose=10)

#### Важности фичей:

In [46]:
imps = model.get_feature_importance(type="PredictionValuesChange")
pairs = sorted(zip(model.feature_names_, imps), key=lambda x: x[1], reverse=True)

for name, fstr in pairs:
    print(f"{name}: {fstr}")


item_popularity: 98.2943027762265
artist_like_cnt: 1.2738894667303473
user_artist_like_cnt: 0.43180775704315116


### 🔍  Retrieval:

#### Кандидатогенератор популярных треков

In [53]:
popular_tracks2 = (
    get_event_oldness(data_part2)
    .unique(["item_id", "uid"])
    .group_by("item_id")
    .agg(pl.col("decay_factor").sum().alias("popularity_decayed"))
    .sort("popularity_decayed", descending=True)[:100]
    .select(["item_id"])
)


debug_test_users = data_part3.select(["uid"]).unique(subset=["uid"])
positive_interactions = debug_test_users.join(
    data_part3.filter(pl.col("target").eq(pl.lit(1))), on="uid"
).select(["uid", "item_id"])

test = debug_test_users.select(["uid"]).join(popular_tracks2, how="cross")
test = add_features(data_part2, test, artists, add_negatives=False, filter_likes=False)

#### Применяем модель

In [54]:
test_pool = Pool(
    data=test.drop(["item_id", "uid"]),
)

In [55]:
scores = model.predict_proba(test_pool)[:, 1]
scores_pl = (
    test.select(["uid", "item_id"])
    .with_columns(pl.Series("score", scores))
    .sort(["uid", "score"], descending=[False, True])
    .with_columns(
        [
            pl.col("score")
            .rank(method="ordinal", descending=True)
            .over("uid")
            .alias("rank")
        ]
    )
    .sort(["uid", "rank"])
)

for k in [100, 300, 1000]:
    recall_k = recall_at_k(
        positive_interactions=positive_interactions,
        candidates=scores_pl,
        k=k,
    )
    print(f"Recall@{k}: {recall_k}")

Recall@100: 0.06109163255531212
Recall@300: 0.06109163255531215
Recall@1000: 0.061091632555312136


In [ ]:
pred = model.predict_proba(test_pool)[:, 1]

submit = (
    test.select(["uid", "item_id"])
    .with_columns(pl.Series("pred", pred))
    .sort(["uid", "pred"], descending=[False, True])
    .group_by("uid")
    .agg(pl.col("item_id").head(100).cast(pl.Utf8).str.join(" ").alias("item_ids"))
)

# submit

uid,item_ids
i64,str
89,"""2960877 7968587 4453817 982903…"
153,"""2960877 7968587 4453817 639925…"
164,"""2960877 7968587 4453817 807749…"
216,"""2960877 7968587 2766576 934228…"
291,"""2960877 7968587 4453817 639925…"
…,…
999735,"""2960877 7968587 4453817 148702…"
999737,"""3167568 2960877 7968587 148702…"
999779,"""2960877 7968587 4453817 639925…"


In [ ]:
submit.write_csv("catboost(5).csv")

### Что дальше?

- Правильная оффлайн валидация (За какие даты собран тест?)
- Правильно собранный пул для обучения
- Больше фичей (Как сделать фичи из эмбеддингов?)
- Более богатые негативы
- Более богатые кандидатогенераторы
- CatBoostClassifier?
- Гиперпараметры модели
- Больше данных
- Учиться на всех данных
<center><img src="https://github.com/yandexdataschool/recsys_course/blob/2026_spring/week02_candgen/Timesplit2.svg?raw=1" width="1100" /></center>